# Full Comparative Pipeline — TripAdvisor Information Retrieval
## Models A → G: From BM25 Baseline to Triple-RRF + Cross-Encoder

This notebook consolidates the **complete model comparison pipeline**, running all models sequentially and printing a final comparison table.

### Models covered
| Model | Technique |
|---|---|
| **Baseline** | BM25 probabilistic ranking |
| **Model A** | TF-IDF + Cosine Similarity |
| **Model B** | MiniLM dense embeddings (keyword bag) |
| **Model C** | MiniLM dense embeddings (raw text, first 500 chars) |
| **Model D** | Score fusion: 70% BM25 + 30% Transformer |
| **Model E** | Two-stage re-ranking: BM25 retrieval → Transformer re-rank |
| **Model F** ⭐ | Score fusion: 85% BM25 + 15% Transformer *(best)* |
| **Model G** | Triple-RRF (BM25 + TF-IDF + mpnet) + Cross-Encoder re-ranking |

### Key design choices
- **Type-Aware Level 2 evaluation**: subcategories are extracted per `typeR` (Hotels → `priceRange`, Restaurants → `restaurantType`+`cuisine`, Attractions → `activiteSubType`), preventing spurious cross-type matches.
- **Corpus-wide TF-IDF normalization**: word weights are relative to the full corpus of 1,835 places, not per individual place.
- **Fixed seed `np.random.seed(42)`**: ensures reproducibility across all runs.

---
## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import time
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

warnings.filterwarnings('ignore')

# Accumulate all results here for the final comparison table
RESULTS = {}

---
## 1. Data Preparation

A **corpus-aware** TF-IDF is fitted on *all* 1,835 places simultaneously (`max_features=5000`), so word weights reflect importance *relative to the whole corpus* — not just within a single place's reviews. This is more discriminative than fitting per-place.

In [ ]:
df_reviews_raw = pd.read_csv('../data/reviews83325.csv')
df_places_meta = pd.read_csv('Tripadvisor.csv', low_memory=False)

print(f"Total reviews: {len(df_reviews_raw)}")
print(f"Total places: {len(df_places_meta)}")

# --- Filter: English reviews only ---
df_reviews_en = df_reviews_raw[df_reviews_raw['langue'] == 'en'].copy()
print(f"English reviews: {len(df_reviews_en)}")

# --- Group all reviews per place into one text block ---
df_grouped = df_reviews_en.groupby('idplace')['review'].apply(
    lambda x: ' '.join(x.dropna().astype(str))
).reset_index()
print(f"Places with English reviews: {len(df_grouped)}")

# --- Corpus-Wide TF-IDF normalization (top 100 words per place) ---
# The vectorizer is fitted on ALL places at once (max_features=5000 global vocabulary)
print("\nFitting corpus-wide TF-IDF vectorizer...")
corpus_vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
all_texts = df_grouped['review'].fillna('').tolist()
corpus_vectorizer.fit(all_texts)
feature_names = corpus_vectorizer.get_feature_names_out()

def extract_top_100_tfidf(text):
    """Transform one place's text through the corpus-fitted vectorizer and return top 100 words."""
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ""
    try:
        X = corpus_vectorizer.transform([text])
        scores = list(zip(feature_names, X.toarray().ravel()))
        sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)
        top_words = [w for w, s in sorted_scores if s > 0][:100]
        return " ".join(top_words) if top_words else text
    except Exception:
        return text

df_grouped['top_100_words'] = df_grouped['review'].apply(extract_top_100_tfidf)
print(f"Sample keywords for place 0: {df_grouped['top_100_words'].iloc[0][:100]}...")

# Save prepared data
df_grouped.to_csv('prepared_reviews_v2.csv', index=False)
print("Saved as 'prepared_reviews_v2.csv'")

FileNotFoundError: [Errno 2] No such file or directory: 'data/reviews83325.csv'

---
## 2. Shared Setup: Split, Evaluation Functions

All models use the **exact same 50/50 train/test split** (seed=42) and the **same evaluation functions** for fair comparison.

In [ ]:
# --- Load & merge ---
df_reviews = pd.read_csv('prepared_reviews_v2.csv')
eval_cols = ['id', 'typeR', 'activiteSubType', 'restaurantType', 'restaurantTypeCuisine', 'priceRange']
df_places = df_places_meta[eval_cols].copy()

df_merged = pd.merge(df_reviews, df_places, left_on='idplace', right_on='id', how='inner')
print(f"Total valid places: {len(df_merged)}")

# --- Deterministic 50/50 split ---
np.random.seed(42)
shuffled_indices = np.random.permutation(len(df_merged))
split_idx = len(df_merged) // 2

train_df = df_merged.iloc[shuffled_indices[:split_idx]].copy().reset_index(drop=True)
test_df  = df_merged.iloc[shuffled_indices[split_idx:]].copy().reset_index(drop=True)

print(f"Train (queries): {len(train_df)} | Test (search space): {len(test_df)}")

# -----------------------------------------------------------------------
# EVALUATION FUNCTIONS
# -----------------------------------------------------------------------

def eval_level_1(query_typeR, sorted_test_typeR_list):
    """Ranking Error L1: position (0-indexed) of first result with matching typeR."""
    if pd.isna(query_typeR) or query_typeR not in sorted_test_typeR_list:
        return None
    for i, t in enumerate(sorted_test_typeR_list):
        if t == query_typeR:
            return i
    return None

def extract_subcategories(row):
    """
    TYPE-AWARE subcategory extraction.
    Only extracts the attribute relevant to the place's type:
      H  (Hotel)          -> priceRange
      R  (Restaurant)     -> restaurantType + restaurantTypeCuisine
      A / AP (Attraction) -> activiteSubType
    This prevents spurious cross-type matches in Level 2 evaluation.
    """
    cats = set()
    type_r = str(row.get('typeR', '')).strip()
    if type_r == 'H':
        val = row.get('priceRange', '')
        if pd.notna(val) and str(val).strip():
            cats.update(p.strip().lower() for p in str(val).split(','))
    elif type_r == 'R':
        for col in ['restaurantType', 'restaurantTypeCuisine']:
            val = row.get(col, '')
            if pd.notna(val) and str(val).strip():
                cats.update(p.strip().lower() for p in str(val).split(','))
    elif type_r in ('A', 'AP'):
        val = row.get('activiteSubType', '')
        if pd.notna(val) and str(val).strip():
            cats.update(p.strip().lower() for p in str(val).split(','))
    return cats

# Pre-compute subcategories for all test places (used by every model)
test_subcats_list = [extract_subcategories(row) for _, row in test_df.iterrows()]

def eval_level_2(query_subcats, sorted_test_indices):
    """Ranking Error L2: position of first result sharing at least one subcategory."""
    if not query_subcats:
        return None
    for i, test_idx in enumerate(sorted_test_indices):
        if query_subcats.intersection(test_subcats_list[test_idx]):
            return i
    return None

def run_evaluation(ranked_indices_per_query, model_name):
    """Generic evaluation loop given a list of ranked test indices per query."""
    lvl1_errors, lvl2_errors = [], []
    test_type_array = test_df['typeR'].values
    t0 = time.time()
    for i, ranked_indices in enumerate(ranked_indices_per_query):
        row = train_df.iloc[i]
        err_1 = eval_level_1(row['typeR'], test_type_array[ranked_indices].tolist())
        if err_1 is not None: lvl1_errors.append(err_1)
        err_2 = eval_level_2(extract_subcategories(row), ranked_indices)
        if err_2 is not None: lvl2_errors.append(err_2)
    l1 = round(np.mean(lvl1_errors), 2)
    l2 = round(np.mean(lvl2_errors), 2)
    print(f"Evaluation done in {time.time()-t0:.2f}s")
    print(f"{model_name} — L1: {l1} ({len(lvl1_errors)} queries) | L2: {l2} ({len(lvl2_errors)} queries)")
    RESULTS[model_name] = (l1, l2)
    return l1, l2

print("All shared functions ready.")

---
## 3. Baseline — BM25

**BM25** (*Best Match 25*) is a probabilistic ranking function that extends TF-IDF with:
- **Term frequency saturation**: diminishing returns past a certain frequency (`k1` parameter)
- **Document length normalization**: longer documents aren't automatically favored (`b` parameter)

It operates on the `top_100_words` token lists and serves as our reference baseline.

In [ ]:
print("Building BM25 index...")
test_texts_kw  = test_df['top_100_words'].fillna('').astype(str).tolist()
train_texts_kw = train_df['top_100_words'].fillna('').astype(str).tolist()

tokenized_corpus = [doc.split() for doc in test_texts_kw]
bm25 = BM25Okapi(tokenized_corpus)
print(f"BM25 indexed {len(tokenized_corpus)} documents.")

print(f"\nEvaluating {len(train_df)} queries...")
bm25_rankings = []
for i in range(len(train_df)):
    query_text = str(train_df.iloc[i]['top_100_words']).strip()
    scores = bm25.get_scores(query_text.split())
    bm25_rankings.append(np.argsort(scores)[::-1])

run_evaluation(bm25_rankings, "BM25 Baseline")

---
## 4. Model A — TF-IDF + Cosine Similarity

Instead of BM25's probabilistic scoring, this model vectorizes all places using TF-IDF and measures similarity geometrically via **cosine similarity** between sparse vectors.

The vectorizer is **fitted on train only** (standard ML practice) then applied to both train and test — this mirrors how a deployed model would work (it has only seen training vocabulary).

In [ ]:
print("Building TF-IDF vectors...")
tfidf_vec = TfidfVectorizer()
train_tfidf = tfidf_vec.fit_transform(train_texts_kw)  # fit on train only
test_tfidf  = tfidf_vec.transform(test_texts_kw)        # apply to test
print(f"Vocabulary size: {len(tfidf_vec.vocabulary_)} | Train: {train_tfidf.shape} | Test: {test_tfidf.shape}")

tfidf_sims = cosine_similarity(train_tfidf, test_tfidf)  # (917, 918) matrix

print(f"\nEvaluating {len(train_df)} queries...")
tfidf_rankings = [np.argsort(tfidf_sims[i])[::-1] for i in range(len(train_df))]
run_evaluation(tfidf_rankings, "Model A (TF-IDF + Cosine)")

---
## 5. Model B — Transformer Dense Embeddings (Keyword Bag)

The model `all-MiniLM-L6-v2` maps each place's `top_100_words` to a **384-dimensional dense vector** that captures semantic meaning.

Unlike TF-IDF/BM25 which require exact lexical overlap, dense embeddings can match *"budget lodge"* with *"cheap hotel"* because they were trained to be context-aware (Distributional Hypothesis).

In [ ]:
bi_encoder_mini = SentenceTransformer('all-MiniLM-L6-v2')

print("Encoding test set (keywords)...")
test_vecs_b  = bi_encoder_mini.encode(test_texts_kw,  batch_size=64, show_progress_bar=True)
print("Encoding train set (keywords)...")
train_vecs_b = bi_encoder_mini.encode(train_texts_kw, batch_size=64, show_progress_bar=True)

dense_sims_b = cosine_similarity(train_vecs_b, test_vecs_b)

print(f"\nEvaluating {len(train_df)} queries...")
model_b_rankings = [np.argsort(dense_sims_b[i])[::-1] for i in range(len(train_df))]
run_evaluation(model_b_rankings, "Model B (MiniLM, keyword bag)")

---
## 6. Model C — Contextual Transformer (Raw Text)

**Hypothesis**: feeding MiniLM full prose (grammatical sentences) instead of a keyword bag should improve results since Transformers were pre-trained on sentence-level text.

We use the **first 500 characters** of the raw concatenated reviews to stay within the token limit.

> **Result**: This approach performs *worse* — TripAdvisor reviews often start with irrelevant preambles (*"I visited last Tuesday with my wife..."*). The TF-IDF keyword selection was already doing a better filtering job.

In [ ]:
# Load raw grouped reviews to access full prose text
df_raw_grouped = pd.read_csv('prepared_reviews_v2.csv')
df_merged_raw = pd.merge(df_raw_grouped[['idplace', 'review']], 
                          df_merged[['idplace', 'typeR', 'activiteSubType', 'restaurantType', 
                                     'restaurantTypeCuisine', 'priceRange']],
                          on='idplace', how='inner')

train_raw = df_merged_raw.iloc[shuffled_indices[:split_idx]].reset_index(drop=True)
test_raw  = df_merged_raw.iloc[shuffled_indices[split_idx:]].reset_index(drop=True)

test_texts_raw  = [str(t)[:500] for t in test_raw['review'].fillna('').tolist()]
train_texts_raw = [str(t)[:500] for t in train_raw['review'].fillna('').tolist()]

print("Encoding test set (raw text, 500 chars)...")
test_vecs_c  = bi_encoder_mini.encode(test_texts_raw,  batch_size=64, show_progress_bar=True)
print("Encoding train set (raw text, 500 chars)...")
train_vecs_c = bi_encoder_mini.encode(train_texts_raw, batch_size=64, show_progress_bar=True)

dense_sims_c = cosine_similarity(train_vecs_c, test_vecs_c)

print(f"\nEvaluating {len(train_df)} queries...")
model_c_rankings = [np.argsort(dense_sims_c[i])[::-1] for i in range(len(train_df))]
run_evaluation(model_c_rankings, "Model C (MiniLM, raw text)")

---
## 7. Model D — Hybrid Score Fusion (70% BM25 + 30% Transformer)

BM25 excels at **precision** (exact keyword matching → Level 1), while Transformers add **semantic recall** (synonym bridging → Level 2). Combining both should yield the best of both worlds.

BM25 scores are **min-max normalized** to `[0, 1]` per query before fusion (their raw range is unbounded and incompatible with cosine similarities).

In [ ]:
# Reuse: bm25 index, dense_sims_c (raw text Transformer)
print(f"Evaluating {len(train_df)} queries (70% BM25 + 30% Transformer)...")
t0 = time.time()
model_d_rankings = []

for i in range(len(train_df)):
    query_text = str(train_df.iloc[i]['top_100_words']).strip()
    if not query_text or query_text == 'nan':
        model_d_rankings.append(np.arange(len(test_df)))
        continue
    
    bm25_scores = bm25.get_scores(query_text.split())
    bm25_max = np.max(bm25_scores)
    bm25_norm = (bm25_scores - np.min(bm25_scores)) / (bm25_max - np.min(bm25_scores) + 1e-9) if bm25_max > 0 else bm25_scores
    
    trans_scores = dense_sims_c[i]  # raw text Transformer sims
    hybrid = 0.7 * bm25_norm + 0.3 * trans_scores
    model_d_rankings.append(np.argsort(hybrid)[::-1])

run_evaluation(model_d_rankings, "Model D (70% BM25 + 30% Transformer)")

---
## 8. Model E — Two-Stage Re-ranking (BM25 → Transformer)

The **industry-standard retrieval pipeline** works in two stages:
1. **Stage 1 (Retrieval)**: BM25 quickly narrows the search to the **top 100** most relevant candidates.
2. **Stage 2 (Re-ranking)**: The Transformer re-orders only those 100 candidates by deep semantic similarity.

> **Result**: Slightly worse than score fusion (Model D). Reason: if BM25 fails to include a semantically relevant place in its top-100 (Stage 1), the Transformer never sees it. Score fusion is more robust as it allows the Transformer to *rescue* any document from anywhere in the corpus.

In [ ]:
# Reuse: bm25, train_vecs_b / test_vecs_b (keyword Transformer — more stable than raw text)
TOP_K_RERANK = 100

print(f"Evaluating {len(train_df)} queries (BM25 top-{TOP_K_RERANK} → Transformer re-rank)...")
t0 = time.time()
model_e_rankings = []

for i in range(len(train_df)):
    query_text = str(train_df.iloc[i]['top_100_words']).strip()
    if not query_text or query_text == 'nan':
        model_e_rankings.append(list(range(len(test_df))))
        continue
    
    # Stage 1: BM25 retrieves top K
    bm25_scores = bm25.get_scores(query_text.split())
    top_k_idx = np.argsort(bm25_scores)[-TOP_K_RERANK:][::-1]
    
    # Stage 2: Transformer re-ranks the top K
    q_vec = train_vecs_b[i].reshape(1, -1)
    rerank_sims = cosine_similarity(q_vec, test_vecs_b[top_k_idx]).flatten()
    reranked_top = top_k_idx[np.argsort(rerank_sims)[::-1]]
    
    # Append remaining (outside top K) in BM25 order
    remaining = [idx for idx in np.argsort(bm25_scores)[::-1] if idx not in set(top_k_idx)]
    model_e_rankings.append(list(reranked_top) + remaining)

run_evaluation(model_e_rankings, "Model E (Two-Stage: BM25→Transformer)")

---
## 9. Model F ⭐ — Optimized Score Fusion (85% BM25 + 15% Transformer)

After diagnosing that Model D (70/30) improved Level 2 but degraded Level 1, we tilt the weight further toward BM25 to **recover Level 1 precision** while retaining the Level 2 semantic gain.

This is the **best-performing model** across both evaluation levels.

In [ ]:
# Reuse: bm25 index, dense_sims_c (raw text Transformer)
print(f"Evaluating {len(train_df)} queries (85% BM25 + 15% Transformer)...")
t0 = time.time()
model_f_rankings = []

for i in range(len(train_df)):
    query_text = str(train_df.iloc[i]['top_100_words']).strip()
    if not query_text or query_text == 'nan':
        model_f_rankings.append(np.arange(len(test_df)))
        continue
    
    bm25_scores = bm25.get_scores(query_text.split())
    bm25_max = np.max(bm25_scores)
    bm25_norm = (bm25_scores - np.min(bm25_scores)) / (bm25_max - np.min(bm25_scores) + 1e-9) if bm25_max > 0 else bm25_scores
    
    trans_scores = dense_sims_c[i]
    hybrid = 0.85 * bm25_norm + 0.15 * trans_scores  # conservative tilt toward BM25
    model_f_rankings.append(np.argsort(hybrid)[::-1])

run_evaluation(model_f_rankings, "Model F ⭐ (85% BM25 + 15% Transformer)")

---
## 10. Model G — Triple-RRF + Cross-Encoder Re-ranking

### Reciprocal Rank Fusion (Cormack et al., 2009)
Instead of normalizing scores (fragile — one outlier can compress the whole distribution), **RRF fuses the *ranks*** from multiple retrieval systems:

$$\text{RRF}(d) = \sum_{r \in \text{rankers}} \frac{1}{k + \text{rank}_r(d)}, \quad k=60$$

This is distribution-agnostic and used in production by Elasticsearch, Pinecone, and most major IR platforms.

### Cross-Encoder Re-ranking
A **bi-encoder** (Models B/C) embeds query and document *independently*. A **cross-encoder** takes both as a *single input*, allowing full attention between every token of both texts simultaneously — far more accurate but ~100× slower (acceptable since we only re-rank the top 50 RRF candidates).

### Architecture
```
Stage 1 — Triple RRF:
  BM25   → rank list 1
  TF-IDF → rank list 2
  mpnet  → rank list 3  (all-mpnet-base-v2: 12 layers, 768-dim)
  RRF fusion → top 50 candidates

Stage 2 — Cross-Encoder:
  ms-marco-MiniLM-L-6-v2 scores each (query, candidate) pair
  Final ranking by cross-encoder score
```

> ⚠️ **This cell will take ~15-20 minutes** due to the mpnet encoding + Cross-Encoder inference on 917 × 50 pairs.

In [ ]:
def reciprocal_rank_fusion(*rank_lists, k=60):
    """Cormack et al. (2009): fuse N ranked lists by reciprocal rank score."""
    rrf_scores = {}
    for rank_list in rank_lists:
        for rank, doc_idx in enumerate(rank_list):
            rrf_scores[doc_idx] = rrf_scores.get(doc_idx, 0.0) + 1.0 / (k + rank)
    return sorted(rrf_scores.keys(), key=lambda d: rrf_scores[d], reverse=True)

# Stage 1c: Dense encoder (upgraded: all-mpnet-base-v2, 12 layers, 768-dim)
print("[Stage 1c] Encoding with all-mpnet-base-v2 (this takes a few minutes)...")
bi_encoder_mpnet = SentenceTransformer('all-mpnet-base-v2')
train_vecs_mpnet = bi_encoder_mpnet.encode(train_texts_kw, batch_size=64, show_progress_bar=True)
test_vecs_mpnet  = bi_encoder_mpnet.encode(test_texts_kw,  batch_size=64, show_progress_bar=True)
dense_sims_g = cosine_similarity(train_vecs_mpnet, test_vecs_mpnet)

# Stage 2: Cross-Encoder
print("\n[Stage 2] Loading Cross-Encoder (ms-marco-MiniLM-L-6-v2)...")
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

TOP_K_CE = 50
print(f"\nEvaluating {len(train_df)} queries (Triple-RRF + Cross-Encoder top {TOP_K_CE})...")
t0 = time.time()
model_g_rankings = []

for i in range(len(train_df)):
    query_text = str(train_df.iloc[i]['top_100_words']).strip()
    if not query_text or query_text == 'nan':
        model_g_rankings.append(list(range(len(test_df))))
        continue
    
    # Rank lists from each retriever
    bm25_ranked  = np.argsort(bm25.get_scores(query_text.split()))[::-1].tolist()
    tfidf_ranked = np.argsort(tfidf_sims[i])[::-1].tolist()
    dense_ranked = np.argsort(dense_sims_g[i])[::-1].tolist()
    
    # RRF fusion
    rrf_ranked = reciprocal_rank_fusion(bm25_ranked, tfidf_ranked, dense_ranked, k=60)
    top_k_idx = rrf_ranked[:TOP_K_CE]
    
    # Cross-Encoder re-ranking
    pairs = [(query_text, test_texts_kw[idx]) for idx in top_k_idx]
    ce_scores = cross_encoder.predict(pairs, show_progress_bar=False)
    reranked_top = [top_k_idx[j] for j in np.argsort(ce_scores)[::-1]]
    
    full_ranked = reranked_top + rrf_ranked[TOP_K_CE:]
    model_g_rankings.append(full_ranked)
    
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(train_df)} queries processed...")

run_evaluation(model_g_rankings, "Model G (Triple-RRF + Cross-Encoder)")

---
## 11. Final Results — Full Comparison Table

In [ ]:
print("\n" + "="*65)
print("FINAL COMPARISON TABLE (Type-Aware Evaluation, seed=42)")
print("="*65)
print(f"{'Model':<45} {'L1 Error':>10} {'L2 Error':>10}")
print("-"*65)
for name, (l1, l2) in RESULTS.items():
    star = " ⭐" if "⭐" in name or l2 == min(v[1] for v in RESULTS.values()) else ""
    print(f"{name:<45} {l1:>10.2f} {l2:>10.2f}{star}")
print("="*65)
print("Lower is better. L1 = typeR match, L2 = subcategory match.")